# Ru MT-Bench: official FastChat judge loop

Эта версия сделана как строгий академический контур оценки, максимально близкий к рекомендациям `t-tech/ru-mt-bench` и LMSYS FastChat:

1. Русский benchmark берётся из `t-tech/ru-mt-bench`.
2. Данные приводятся к стандартному FastChat layout `fastchat/llm_judge/data/mt_bench/...`.
3. Ответы наших LoRA/Unsloth моделей генерируются в официальном формате `model_answer/<model_id>.jsonl`.
4. Судейство и итоговая таблица выполняются официальными FastChat скриптами `gen_judgment.py` и `show_result.py` без изменения их prompt/rubric.
5. Собственная логика используется только как compatibility shim для генерации ответов из Unsloth QLoRA adapters, потому что официальный `gen_model_answer.py` не знает наши adapter dirs и фиксированный Qwen2.5 ChatML.


## Methodological contract

Что считается официальным в этой тетрадке:

- judge prompts: `FastChat/fastchat/llm_judge/data/judge_prompts.jsonl`;
- judge runner: `FastChat/fastchat/llm_judge/gen_judgment.py`;
- result display: `FastChat/fastchat/llm_judge/show_result.py`;
- MT-Bench answer schema: `question_id`, `answer_id`, `model_id`, `choices`, `tstamp`.

Что не является официальным и почему:

- генератор ответов для наших моделей написан здесь, потому что модели лежат как Unsloth LoRA adapters в Google Drive;
- Qwen форматируется через явный `unsloth.chat_templates.get_chat_template(..., "qwen-2.5")`, а не через стоковый `tokenizer.apply_chat_template` без фикса шаблона.

Важно: `BENCH_NAME_FOR_FASTCHAT = "mt_bench"` выбран намеренно. В официальном `show_result.py` второй turn печатается только для `bench_name == "mt_bench"`, поэтому русские вопросы кладутся в стандартную директорию `data/mt_bench`, а фактический источник фиксируется в metadata как `t-tech/ru-mt-bench`.
Для ref-based категорий FastChat (`math`, `reasoning`, `coding`) официальный `gen_judgment.py` требует `reference_answer/<judge>.jsonl`. Поэтому тетрадка берёт вопросы и references из HF split `t-tech/ru-mt-bench`, пишет `question.jsonl` в FastChat-формате и отдельно пишет `reference_answer/{JUDGE_MODEL}.jsonl`. Raw `question.jsonl` один сам по себе для строгого official judge loop недостаточен.


In [ ]:
# --- 0. Configuration ---

USE_DRIVE = True
OUTPUT_BASE_OVERRIDE = None  # например: "/content/gdrive/MyDrive/diploma"

FASTCHAT_DIR = "/content/FastChat"
FASTCHAT_REF = "main"  # для финального отчёта можно заменить на конкретный commit SHA
BENCH_HF_DATASET = "t-tech/ru-mt-bench"
BENCH_NAME_FOR_FASTCHAT = "mt_bench"  # см. methodological contract выше

MODEL_FILTER = None  # например: ["baseline_masked_packed_base_90k_1ep_noleak", "entropy_90k_from_200k_noleak"]
MAX_QUESTIONS = None  # None = все 80 вопросов; для smoke test можно поставить 3
FIRST_N_JUDGMENTS = None  # None = все judgment; для smoke test можно поставить 6

OVERWRITE_ANSWERS = False
OVERWRITE_JUDGMENTS = True  # official gen_judgment append-only, поэтому чистый запуск безопаснее

MAX_SEQ_LEN = 2048
MAX_NEW_TOKENS = 1024  # official gen_model_answer.py default
NUM_CHOICES = 1        # official default

JUDGE_MODEL = "gpt-4"  # official MT-Bench default judge model
JUDGE_PARALLEL = 2
JUDGMENT_MODE = "single"  # official recommended MT-Bench default



In [ ]:
# --- 1. Install dependencies and clone FastChat (Colab shell commands) ---
# Если зависимости уже установлены в этой runtime, эту ячейку можно не запускать.
# Держим путь к FastChat захардкоженным, чтобы install cell не зависела от Python state.

!python -m pip install -q --upgrade datasets "pandas==2.2.2" tqdm tabulate shortuuid huggingface_hub openai anthropic unsloth bitsandbytes accelerate sentencepiece
![ -d "/content/FastChat/.git" ] || git clone https://github.com/lm-sys/FastChat.git "/content/FastChat"
!git -C "/content/FastChat" fetch --all --tags
!git -C "/content/FastChat" checkout "main"
!python -m pip install -q -e "/content/FastChat[model_worker,llm_judge]"
# Colab pins pandas to 2.2.2; keep it pinned after editable FastChat installs.
!python -m pip install -q --upgrade "pandas==2.2.2"


In [ ]:
# --- 1b. Register FastChat source and capture metadata ---

import os, sys, subprocess, json, hashlib, time, shutil, glob, gc, re
from pathlib import Path

FASTCHAT_DIR = globals().get("FASTCHAT_DIR", "/content/FastChat")
FASTCHAT_REF = globals().get("FASTCHAT_REF", "main")
BENCH_NAME_FOR_FASTCHAT = globals().get("BENCH_NAME_FOR_FASTCHAT", "mt_bench")


def run(cmd, cwd=None, input_text=None, check=True):
    print("$", " ".join(map(str, cmd)))
    result = subprocess.run(
        list(map(str, cmd)),
        cwd=cwd,
        input=input_text,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with code {result.returncode}: {' '.join(map(str, cmd))}")
    return result


if FASTCHAT_DIR not in sys.path:
    sys.path.insert(0, FASTCHAT_DIR)

try:
    import fastchat  # noqa: F401
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        f"fastchat не импортируется. Сначала запусти предыдущую ячейку установки. "
        f"Проверяемый путь: {FASTCHAT_DIR}"
    ) from exc

FASTCHAT_COMMIT = subprocess.check_output(
    ["git", "-C", FASTCHAT_DIR, "rev-parse", "HEAD"], text=True
).strip()
LLM_JUDGE_DIR = Path(FASTCHAT_DIR) / "fastchat" / "llm_judge"
FASTCHAT_DATA_DIR = LLM_JUDGE_DIR / "data" / BENCH_NAME_FOR_FASTCHAT

print("FASTCHAT_COMMIT =", FASTCHAT_COMMIT)
print("LLM_JUDGE_DIR   =", LLM_JUDGE_DIR)
print("FASTCHAT_DATA   =", FASTCHAT_DATA_DIR)
print("fastchat import path OK")


In [ ]:
# --- 2. Drive mount and output directory discovery ---

import os, glob
from pathlib import Path


def _is_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def _is_drive_mounted(path):
    return os.path.isdir(os.path.join(path, "MyDrive"))


def _mount_drive_parallel_safe():
    from google.colab import drive
    mountpoints = ["/content/drive", "/content/gdrive"] + [f"/content/drive_{i}" for i in range(1, 8)]

    for mountpoint in mountpoints:
        if _is_drive_mounted(mountpoint):
            print(f"Google Drive уже доступен: {mountpoint}")
            return mountpoint

    last_error = None
    for mountpoint in mountpoints:
        try:
            if os.path.exists(mountpoint):
                if not os.path.isdir(mountpoint):
                    print(f"{mountpoint} занят файлом — пробуем следующий")
                    continue
                if os.listdir(mountpoint):
                    print(f"{mountpoint} занят локальными файлами — пробуем следующий")
                    continue
            os.makedirs(mountpoint, exist_ok=True)
            drive.mount(mountpoint)
            return mountpoint
        except ValueError as exc:
            last_error = exc
            print(f"{mountpoint}: {exc}")
            if _is_drive_mounted(mountpoint):
                return mountpoint
    raise RuntimeError("Не удалось подключить Google Drive. Задай OUTPUT_BASE_OVERRIDE.") from last_error


def _count_adapters(base):
    return len(glob.glob(f"{base}/outputs/*/adapter/adapter_config.json"))


def choose_output_base():
    if OUTPUT_BASE_OVERRIDE:
        return OUTPUT_BASE_OVERRIDE
    if USE_DRIVE and _is_colab():
        _mount_drive_parallel_safe()
        mountpoints = ["/content/drive", "/content/gdrive"] + [f"/content/drive_{i}" for i in range(1, 8)]
        candidates = [f"{m}/MyDrive/diploma" for m in mountpoints]
    else:
        candidates = ["."]
    existing = [c for c in candidates if os.path.isdir(f"{c}/outputs")]
    if not existing:
        raise RuntimeError("Не найден diploma/outputs. Проверь Drive или OUTPUT_BASE_OVERRIDE.")
    ranked = sorted(existing, key=lambda c: (_count_adapters(c), c), reverse=True)
    print("OUTPUT_BASE candidates:")
    for c in ranked:
        print(f"  adapters={_count_adapters(c):2d}  {c}")
    return ranked[0]


OUTPUT_BASE = choose_output_base()
OUTPUTS_DIR = f"{OUTPUT_BASE}/outputs"
EVAL_DIR = f"{OUTPUT_BASE}/ru_mt_bench_fastchat_official"
REPORTS_DIR = f"{EVAL_DIR}/reports"
ARTIFACTS_DIR = f"{EVAL_DIR}/fastchat_artifacts"
for p in [EVAL_DIR, REPORTS_DIR, ARTIFACTS_DIR]:
    os.makedirs(p, exist_ok=True)

print(f"\nOUTPUT_BASE = {OUTPUT_BASE}")
print(f"EVAL_DIR    = {EVAL_DIR}")


In [ ]:
# --- 3. Discover trained adapters ---

import pandas as pd

MODEL_FILTER = globals().get("MODEL_FILTER", None)


def load_json_if_exists(path):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        return {}


def discover_adapters(outputs_dir):
    rows = []
    for cfg in sorted(Path(outputs_dir).glob("*/adapter/adapter_config.json")):
        strategy_dir = cfg.parents[1]
        model_id = strategy_dir.name
        if MODEL_FILTER is not None and model_id not in MODEL_FILTER:
            continue
        metrics = load_json_if_exists(strategy_dir / "final_metrics.json")
        rows.append({
            "model_id": model_id,
            "adapter_dir": str(strategy_dir / "adapter"),
            "strategy_dir": str(strategy_dir),
            "has_final_metrics": bool(metrics),
            "ppl_common": metrics.get("perplexity_assistant_only_on_common_val") or metrics.get("perplexity_assistant_only"),
            "ppl_own": metrics.get("perplexity_assistant_only_on_own_val"),
            "full_text_ppl": metrics.get("perplexity"),
        })
    priority = {
        "baseline_masked_packed_base_90k_1ep_noleak": 0,
        "entropy_90k_from_200k_noleak": 1,
        "loss_rho_90k_from_200k_noleak": 2,
        "ifd_90k_from_200k_noleak": 3,
        "quality_classifier_90k_from_200k_noleak": 4,
    }
    return pd.DataFrame(sorted(rows, key=lambda r: (priority.get(r["model_id"], 99), r["model_id"])))


models_df = discover_adapters(OUTPUTS_DIR)
if models_df.empty:
    raise RuntimeError(f"Не найдено адаптеров: {OUTPUTS_DIR}/*/adapter/adapter_config.json")
MODEL_LIST = models_df["model_id"].tolist()
print(f"Найдено моделей: {len(models_df)}")
display(models_df)


In [ ]:
# --- 4. Prepare ru-mt-bench in official FastChat data layout ---

from datasets import load_dataset

BENCH_HF_DATASET = globals().get("BENCH_HF_DATASET", "t-tech/ru-mt-bench")
BENCH_NAME_FOR_FASTCHAT = globals().get("BENCH_NAME_FOR_FASTCHAT", "mt_bench")
FASTCHAT_DIR = globals().get("FASTCHAT_DIR", "/content/FastChat")
JUDGE_MODEL = globals().get("JUDGE_MODEL", "gpt-4")
MAX_QUESTIONS = globals().get("MAX_QUESTIONS", None)

# Make reruns robust even if editable pip install did not refresh Colab import paths.
if FASTCHAT_DIR not in sys.path:
    sys.path.insert(0, FASTCHAT_DIR)
from fastchat.llm_judge.common import NEED_REF_CATS
import shortuuid


def normalize_turns(value):
    if value is None:
        return []
    if hasattr(value, "tolist"):
        value = value.tolist()
    if isinstance(value, tuple):
        value = list(value)
    if not isinstance(value, list):
        return []
    return ["" if x is None else str(x) for x in value]


bench_ds = load_dataset(BENCH_HF_DATASET, split="train")
if MAX_QUESTIONS is not None:
    bench_ds = bench_ds.select(range(min(MAX_QUESTIONS, len(bench_ds))))
bench_df = pd.DataFrame(bench_ds)

if "reference" not in bench_df.columns:
    raise RuntimeError(
        "В загруженном split нет колонки reference. Для строгого FastChat judge нужны reference answers "
        "для категорий math/reasoning/coding. Не используй raw/question.jsonl без reference_answer."
    )

question_rows = []
ref_rows = []
missing_refs = []
for row in bench_df.to_dict("records"):
    qid = int(row["question_id"])
    category = str(row["category"])
    turns = normalize_turns(row["turns"])
    refs = normalize_turns(row.get("reference"))
    question_rows.append({"question_id": qid, "category": category, "turns": turns})

    if category in set(NEED_REF_CATS):
        if len(refs) < len(turns):
            missing_refs.append((qid, category, refs))
        else:
            ref_rows.append({
                "question_id": qid,
                "answer_id": shortuuid.uuid(),
                "model_id": JUDGE_MODEL,
                "choices": [{"index": 0, "turns": refs}],
                "tstamp": time.time(),
            })

if missing_refs:
    raise RuntimeError(f"Не хватает reference answers для FastChat ref-based categories: {missing_refs[:10]}")

question_dir = FASTCHAT_DATA_DIR
model_answer_dir = question_dir / "model_answer"
reference_answer_dir = question_dir / "reference_answer"
model_judgment_dir = question_dir / "model_judgment"
for p in [question_dir, model_answer_dir, reference_answer_dir, model_judgment_dir]:
    p.mkdir(parents=True, exist_ok=True)

question_file = question_dir / "question.jsonl"
reference_file = reference_answer_dir / f"{JUDGE_MODEL}.jsonl"
with open(question_file, "w", encoding="utf-8") as f:
    for row in question_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
with open(reference_file, "w", encoding="utf-8") as f:
    for row in ref_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Dataset: {BENCH_HF_DATASET}; rows={len(question_rows)}")
print(f"Questions:  {question_file}")
print(f"References: {reference_file} ({len(ref_rows)} rows for {sorted(set(NEED_REF_CATS))})")
display(bench_df["category"].value_counts().sort_index().to_frame("n"))
display(bench_df.head(3))


In [ ]:
# --- 5. Record official source hashes before running evaluation ---


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


source_files = {
    "gen_judgment.py": str(LLM_JUDGE_DIR / "gen_judgment.py"),
    "show_result.py": str(LLM_JUDGE_DIR / "show_result.py"),
    "common.py": str(LLM_JUDGE_DIR / "common.py"),
    "judge_prompts.jsonl": str(LLM_JUDGE_DIR / "data" / "judge_prompts.jsonl"),
    "question.jsonl": str(question_file),
    f"reference_answer/{JUDGE_MODEL}.jsonl": str(reference_file),
}
source_hashes = {name: sha256_file(path) for name, path in source_files.items()}
print(json.dumps(source_hashes, indent=2, ensure_ascii=False))


In [ ]:
# --- 6. Generate model answers in official FastChat answer schema ---

import torch
from tqdm.auto import tqdm

MAX_SEQ_LEN = globals().get("MAX_SEQ_LEN", 2048)
MAX_NEW_TOKENS = globals().get("MAX_NEW_TOKENS", 1024)
NUM_CHOICES = globals().get("NUM_CHOICES", 1)
OVERWRITE_ANSWERS = globals().get("OVERWRITE_ANSWERS", False)

import unsloth  # важно импортировать до transformers
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from fastchat.llm_judge.common import temperature_config
from transformers import GenerationConfig

def detect_bf16_or_stop():
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA недоступна. Нужен GPU runtime.")
    major, minor = torch.cuda.get_device_capability(0)
    if major < 8:
        raise RuntimeError(f"bf16 недоступен на GPU sm_{major}{minor}; нужен A100/L4/H100 или другой sm_80+.")
    return torch.bfloat16


DTYPE = detect_bf16_or_stop()
print("GPU:", torch.cuda.get_device_name(0))
print("dtype:", DTYPE)


def answer_file_for(model_id):
    return model_answer_dir / f"{model_id}.jsonl"


def read_jsonl(path):
    if not os.path.isfile(path):
        return []
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def append_jsonl(path, row):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


def reorg_answer_file(path):
    answers = {}
    for line in read_jsonl(path):
        answers[int(line["question_id"])] = json.dumps(line, ensure_ascii=False) + "\n"
    with open(path, "w", encoding="utf-8") as f:
        for qid in sorted(answers):
            f.write(answers[qid])


def clean_response(tokenizer, text):
    text = (text or "").strip()
    for stop in ["<|im_end|>", "<|endoftext|>", "<|end_of_text|>"]:
        if stop in text:
            text = text.split(stop, 1)[0].strip()
    for special in tokenizer.special_tokens_map.values():
        specials = special if isinstance(special, list) else [special]
        for item in specials:
            if item:
                text = text.replace(item, "")
    return text.strip()


def build_messages(history):
    messages = []
    for user_text, assistant_text in history:
        messages.append({"role": "user", "content": user_text})
        if assistant_text is not None:
            messages.append({"role": "assistant", "content": assistant_text})
    return messages


@torch.inference_mode()
def generate_turn(model, tokenizer, history, category):
    prompt = tokenizer.apply_chat_template(build_messages(history), tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([prompt], return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).to(model.device)
    temperature = temperature_config.get(category, 0.7)
    do_sample = temperature >= 1e-4
    gen_kwargs = dict(
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=do_sample,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )
    if do_sample:
        gen_kwargs["temperature"] = temperature
    output_ids = model.generate(**inputs, **gen_kwargs)
    output_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return clean_response(tokenizer, tokenizer.decode(output_ids, spaces_between_special_tokens=False))


def generate_for_model(model_id, adapter_dir):
    out_path = answer_file_for(model_id)
    if OVERWRITE_ANSWERS and os.path.isfile(out_path):
        os.remove(out_path)
    done = {int(row["question_id"]) for row in read_jsonl(out_path)}
    todo = [row for row in question_rows if int(row["question_id"]) not in done]
    if not todo:
        print(f"{model_id}: official answer file already complete")
        return

    print(f"\n=== {model_id}: load {adapter_dir} ===")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=adapter_dir,
        max_seq_length=MAX_SEQ_LEN,
        dtype=DTYPE,
        load_in_4bit=True,
    )
    tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")
    tokenizer.padding_side = "left"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    FastLanguageModel.for_inference(model)
    # Qwen/Unsloth may carry max_length=32768 in generation_config.
    # We intentionally use max_new_tokens; resetting max_length to the library default
    # removes the duplicate-control warning without changing output length semantics.
    if getattr(model, "generation_config", None) is not None:
        model.generation_config.max_length = GenerationConfig().max_length

    for question in tqdm(todo, desc=f"generate {model_id}"):
        choices = []
        for choice_idx in range(NUM_CHOICES):
            torch.manual_seed(choice_idx)
            history = []
            model_turns = []
            for user_turn in question["turns"]:
                history.append((user_turn, None))
                try:
                    output = generate_turn(model, tokenizer, history, question["category"])
                except RuntimeError as exc:
                    print("ERROR question ID:", question["question_id"], exc)
                    output = "ERROR"
                history[-1] = (user_turn, output)
                model_turns.append(output)
            choices.append({"index": choice_idx, "turns": model_turns})
        append_jsonl(out_path, {
            "question_id": int(question["question_id"]),
            "answer_id": shortuuid.uuid(),
            "model_id": model_id,
            "choices": choices,
            "tstamp": time.time(),
        })

    reorg_answer_file(out_path)
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print(f"{model_id}: saved {out_path}")


In [ ]:
# --- 7. Run answer generation for all selected adapters ---

for row in models_df.itertuples(index=False):
    generate_for_model(row.model_id, row.adapter_dir)

print("Answer files:")
for model_id in MODEL_LIST:
    path = answer_file_for(model_id)
    print(model_id, len(read_jsonl(path)), path)


In [ ]:
# --- 8. Run official FastChat gen_judgment.py unchanged ---

JUDGE_MODEL = globals().get("JUDGE_MODEL", "gpt-4")
JUDGE_PARALLEL = globals().get("JUDGE_PARALLEL", 2)
JUDGMENT_MODE = globals().get("JUDGMENT_MODE", "single")
FIRST_N_JUDGMENTS = globals().get("FIRST_N_JUDGMENTS", None)
OVERWRITE_JUDGMENTS = globals().get("OVERWRITE_JUDGMENTS", True)

if JUDGE_MODEL.startswith("gpt") and not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("Для официального FastChat GPT judge нужен OPENAI_API_KEY в окружении.")
if JUDGE_MODEL.startswith("claude") and not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError("Для официального FastChat Claude judge нужен ANTHROPIC_API_KEY в окружении.")

judgment_file = model_judgment_dir / f"{JUDGE_MODEL}_{'single' if JUDGMENT_MODE == 'single' else 'pair'}.jsonl"
if OVERWRITE_JUDGMENTS and judgment_file.exists():
    judgment_file.unlink()

cmd = [
    sys.executable,
    "gen_judgment.py",
    "--bench-name", BENCH_NAME_FOR_FASTCHAT,
    "--judge-model", JUDGE_MODEL,
    "--mode", JUDGMENT_MODE,
    "--parallel", str(JUDGE_PARALLEL),
    "--model-list", *MODEL_LIST,
]
if FIRST_N_JUDGMENTS is not None:
    cmd += ["--first-n", str(FIRST_N_JUDGMENTS)]

print("Official command:")
print(" ".join(cmd))
result = run(cmd, cwd=str(LLM_JUDGE_DIR), input_text="\n")
print("Judgments:", judgment_file)


In [ ]:
# --- 9. Run official FastChat show_result.py unchanged ---

cmd = [
    sys.executable,
    "show_result.py",
    "--bench-name", BENCH_NAME_FOR_FASTCHAT,
    "--judge-model", JUDGE_MODEL,
    "--mode", JUDGMENT_MODE,
    "--model-list", *MODEL_LIST,
]
print("Official command:")
print(" ".join(cmd))
show_result = run(cmd, cwd=str(LLM_JUDGE_DIR))

show_result_txt = Path(REPORTS_DIR) / "official_show_result.txt"
show_result_txt.write_text(show_result.stdout, encoding="utf-8")
print("Saved official stdout:", show_result_txt)


In [ ]:
# --- 10. Academic metadata and convenience tables derived from official judgments ---

import numpy as np

judgment_rows = read_jsonl(judgment_file)
if not judgment_rows:
    raise RuntimeError(f"Official judgment file is empty: {judgment_file}")

judgments_df = pd.DataFrame(judgment_rows)
question_meta_df = pd.DataFrame(question_rows)[["question_id", "category"]]
judgments_df = judgments_df.merge(question_meta_df, on="question_id", how="left")
judgments_df["score"] = pd.to_numeric(judgments_df["score"], errors="coerce")
valid_df = judgments_df[judgments_df["score"] != -1].dropna(subset=["score"])

summary_df = (
    valid_df.groupby("model")
    .agg(
        official_mt_bench_avg=("score", "mean"),
        official_mt_bench_std=("score", "std"),
        n_judgments=("score", "count"),
        n_questions=("question_id", "nunique"),
    )
    .reset_index()
    .rename(columns={"model": "model_id"})
    .merge(models_df, on="model_id", how="left")
    .sort_values("official_mt_bench_avg", ascending=False)
)
turn_df = (
    valid_df.groupby(["model", "turn"])["score"].mean()
    .unstack("turn")
    .rename(columns={1: "turn_1_score", 2: "turn_2_score"})
    .reset_index()
    .rename(columns={"model": "model_id"})
)
category_df = (
    valid_df.groupby(["model", "category"])["score"].mean()
    .reset_index()
    .rename(columns={"model": "model_id", "score": "category_score"})
)
summary_df = summary_df.merge(turn_df, on="model_id", how="left")

fastchat_artifacts = Path(ARTIFACTS_DIR)
fastchat_artifacts.mkdir(parents=True, exist_ok=True)
shutil.copy2(question_file, fastchat_artifacts / "question.jsonl")
shutil.copy2(reference_file, fastchat_artifacts / f"reference_answer_{JUDGE_MODEL}.jsonl")
shutil.copy2(judgment_file, fastchat_artifacts / judgment_file.name)
for model_id in MODEL_LIST:
    shutil.copy2(answer_file_for(model_id), fastchat_artifacts / f"model_answer_{model_id}.jsonl")

summary_csv = Path(REPORTS_DIR) / "official_summary.csv"
turn_csv = Path(REPORTS_DIR) / "official_turn_scores.csv"
category_csv = Path(REPORTS_DIR) / "official_category_scores.csv"
details_csv = Path(REPORTS_DIR) / "official_judgments_details.csv"
summary_df.to_csv(summary_csv, index=False)
turn_df.to_csv(turn_csv, index=False)
category_df.to_csv(category_csv, index=False)
judgments_df.to_csv(details_csv, index=False)

gen_judgment_cmd = [
    sys.executable, "gen_judgment.py",
    "--bench-name", BENCH_NAME_FOR_FASTCHAT,
    "--judge-model", JUDGE_MODEL,
    "--mode", JUDGMENT_MODE,
    "--parallel", str(JUDGE_PARALLEL),
    "--model-list", *MODEL_LIST,
]
if FIRST_N_JUDGMENTS is not None:
    gen_judgment_cmd += ["--first-n", str(FIRST_N_JUDGMENTS)]
show_result_cmd = [
    sys.executable, "show_result.py",
    "--bench-name", BENCH_NAME_FOR_FASTCHAT,
    "--judge-model", JUDGE_MODEL,
    "--mode", JUDGMENT_MODE,
    "--model-list", *MODEL_LIST,
]

metadata = {
    "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "fastchat_repo": "https://github.com/lm-sys/FastChat",
    "fastchat_ref_requested": FASTCHAT_REF,
    "fastchat_commit": FASTCHAT_COMMIT,
    "bench_hf_dataset": BENCH_HF_DATASET,
    "bench_name_for_fastchat": BENCH_NAME_FOR_FASTCHAT,
    "judge_model": JUDGE_MODEL,
    "judgment_mode": JUDGMENT_MODE,
    "judge_parallel": JUDGE_PARALLEL,
    "first_n_judgments": FIRST_N_JUDGMENTS,
    "max_questions": MAX_QUESTIONS,
    "max_seq_len": MAX_SEQ_LEN,
    "max_new_tokens": MAX_NEW_TOKENS,
    "num_choices": NUM_CHOICES,
    "model_list": MODEL_LIST,
    "official_commands": {
        "gen_judgment": " ".join(map(str, gen_judgment_cmd)),
        "show_result": " ".join(map(str, show_result_cmd)),
    },
    "source_hashes_sha256": source_hashes,
    "files": {
        "question_file": str(question_file),
        "reference_file": str(reference_file),
        "judgment_file": str(judgment_file),
        "official_show_result_stdout": str(show_result_txt),
        "summary_csv": str(summary_csv),
        "turn_csv": str(turn_csv),
        "category_csv": str(category_csv),
        "details_csv": str(details_csv),
        "fastchat_artifacts_dir": str(fastchat_artifacts),
    },
}
metadata_path = Path(REPORTS_DIR) / "official_eval_metadata.json"
metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")

display(summary_df[["model_id", "official_mt_bench_avg", "turn_1_score", "turn_2_score", "n_questions", "n_judgments", "ppl_common"]])
display(category_df.pivot(index="model_id", columns="category", values="category_score").reindex(summary_df["model_id"]))
print("Saved:")
print(summary_csv)
print(turn_csv)
print(category_csv)
print(details_csv)
print(metadata_path)


In [ ]:
# --- 11. Short Markdown report from official outputs ---

report_md = Path(REPORTS_DIR) / "official_ru_mt_bench_report.md"
summary_report = summary_df[["model_id", "official_mt_bench_avg", "turn_1_score", "turn_2_score", "n_questions", "ppl_common"]].copy()
for col in ["official_mt_bench_avg", "turn_1_score", "turn_2_score", "ppl_common"]:
    summary_report[col] = summary_report[col].map(lambda x: "" if pd.isna(x) else f"{x:.4f}")
category_report = category_df.pivot(index="model_id", columns="category", values="category_score").reindex(summary_df["model_id"])
for col in category_report.columns:
    category_report[col] = category_report[col].map(lambda x: "" if pd.isna(x) else f"{x:.3f}")

report = f"""# Official FastChat Ru MT-Bench Evaluation

Generated: `{metadata['created_at']}`

- Dataset: `{BENCH_HF_DATASET}`
- FastChat commit: `{FASTCHAT_COMMIT}`
- FastChat bench dir: `{BENCH_NAME_FOR_FASTCHAT}` containing Russian questions
- Judge model: `{JUDGE_MODEL}`
- Judgment mode: `{JUDGMENT_MODE}`
- Official judge command: `{metadata['official_commands']['gen_judgment']}`
- Official result command: `{metadata['official_commands']['show_result']}`

## Official Average Scores

{summary_report.to_markdown(index=False)}

## Scores By Category

{category_report.reset_index().to_markdown(index=False)}

## Reproducibility Files

- Metadata: `{metadata_path}`
- Official show_result stdout: `{show_result_txt}`
- Raw official judgments: `{judgment_file}`
- Copied FastChat artifacts: `{fastchat_artifacts}`
"""
report_md.write_text(report, encoding="utf-8")
print(report_md)
print(report)
